# Couleurs et palettes du dataset

Ce notebook aide à examiner les couleurs réellement présentes dans un sprite et à appliquer les définitions de [`taxonomies_couleurs.json`](taxonomies_couleurs.json). Il affiche l’image à sa taille native, la fréquence exacte des couleurs opaques et un aperçu des couleurs dominantes. La taxonomie distingue les couleurs observées des propriétés globales de la palette. Les mesures de pixels sont descriptives : elles ne remplacent pas le jugement visuel pour la température, la structure ou le contraste.

Les familles dominantes sont **multi-étiquettes** : plusieurs couleurs peuvent être importantes dans une image. Ne sélectionne pas chaque couleur isolée. La taille de palette est calculée en comptant les couleurs RGB distinctes parmi les pixels dont l’alpha est supérieur à zéro.

In [ ]:
from pathlib import Path
from collections import Counter
import json
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

PROJECT_ROOT = Path.cwd().resolve()
for candidate in (PROJECT_ROOT, *PROJECT_ROOT.parents):
    if (candidate / "notebook/dataset/taxonomy/taxonomies_couleurs.json").is_file():
        PROJECT_ROOT = candidate
        break
TAXONOMY_PATH = PROJECT_ROOT / "notebook/dataset/taxonomy/taxonomies_couleurs.json"
DATASET_ROOT = PROJECT_ROOT / "dataset-viewer/datasets/pixelart-dataset/.gitrepo"
taxonomy = json.loads(TAXONOMY_PATH.read_text(encoding="utf-8"))

# Tu peux renseigner un chemin précis. Sinon, le notebook choisit une image du dataset local.
IMAGE_PATH = None
IMAGE_SUFFIXES = {".png", ".jpg", ".jpeg", ".webp", ".gif"}
if IMAGE_PATH is None:
    image_candidates = sorted(p for p in (DATASET_ROOT / "data").rglob("*")
                              if p.is_file() and p.suffix.lower() in IMAGE_SUFFIXES)
    if not image_candidates:
        raise FileNotFoundError(f"Aucune image trouvée sous {DATASET_ROOT / 'data'}")
    IMAGE_PATH = image_candidates[0]
else:
    IMAGE_PATH = Path(IMAGE_PATH)
    if not IMAGE_PATH.is_absolute():
        IMAGE_PATH = PROJECT_ROOT / IMAGE_PATH

print(f"Image : {IMAGE_PATH.relative_to(PROJECT_ROOT) if IMAGE_PATH.is_relative_to(PROJECT_ROOT) else IMAGE_PATH}")

## Aperçu de l’image et des couleurs

Le comptage est exact sur les pixels opaques du fichier. Les couleurs transparentes ne sont pas transformées en blanc. L’aperçu agrège les couleurs RGB identiques et trie les échantillons par nombre de pixels.

In [ ]:
with Image.open(IMAGE_PATH) as source:
    rgba = np.asarray(source.convert("RGBA"))
height, width = rgba.shape[:2]
pixels = rgba.reshape(-1, 4)
opaque = pixels[pixels[:, 3] > 0, :3]
if opaque.size == 0:
    raise ValueError("L’image ne contient aucun pixel visible.")
counts = Counter(map(tuple, opaque.tolist()))
unique_count = len(counts)
total_visible = int(opaque.shape[0])
ranked = counts.most_common(16)

edges = sorted(taxonomy["hierarchies"]["palette_size"], key=lambda k: ["very_limited", "limited", "moderate", "broad", "very_broad", "unknown", "not_applicable"].index(k))
if unique_count <= 4: size_label = "very_limited"
elif unique_count <= 8: size_label = "limited"
elif unique_count <= 16: size_label = "moderate"
elif unique_count <= 32: size_label = "broad"
else: size_label = "very_broad"
size_description = taxonomy["hierarchies"]["palette_size"][size_label]["description"]

fig, (ax_image, ax_palette) = plt.subplots(1, 2, figsize=(13, 5), gridspec_kw={"width_ratios": [1, 1.3]})
ax_image.imshow(rgba, interpolation="nearest")
ax_image.set_title(f"Image native · {width} × {height} px")
ax_image.axis("off")
ax_palette.barh(range(len(ranked)), [count for _, count in ranked], color=[np.array(rgb) / 255 for rgb, _ in ranked])
ax_palette.set_yticks(range(len(ranked)), [f"#{r:02X}{g:02X}{b:02X}" for (r, g, b), _ in ranked])
ax_palette.invert_yaxis()
ax_palette.set_xlabel("Pixels visibles")
ax_palette.set_title("Jusqu’aux 16 couleurs les plus fréquentes")
ax_palette.grid(axis="x", alpha=.2)
plt.tight_layout()
plt.show()

print(f"{unique_count} couleurs RGB distinctes · {total_visible:,} pixels visibles · classe {size_label}")
print(size_description)

## Définitions destinées à l’annotation

Cette cellule affiche les catégories et leurs descriptions complètes. Elle sert de référence pour préparer une annotation ou vérifier les choix du modèle. Les catégories sous `dominant_color_families` peuvent être combinées ; les autres dimensions reçoivent une valeur unique.

In [ ]:
def print_tree(tree, prefix=""):
    for key, item in tree.items():
        print(f"{prefix}{key}: {item['description']}")
        if item["children"]:
            print_tree(item["children"], prefix + "  ")

for dimension, definition in taxonomy["dimensions"].items():
    print(f"\n## {definition['label']} — {definition['cardinality']}")
    print(definition["definition"])
    print("Applicabilité :", definition["applicability"])
    print_tree(taxonomy["hierarchies"][dimension])

print("\n## Règles de mesure")
for key, description in taxonomy["measurement_guidance"].items():
    print(f"- {key}: {description}")

## Exemple de sortie structurée

`palette_size` peut être préremplie par le comptage ci-dessus. `dominant_color_families` reste une liste, car plusieurs familles peuvent dominer simultanément. Les champs perceptifs doivent être décidés à partir de l’image, pas déduits du seul histogramme RGB.

In [ ]:
example = {
    "source": str(IMAGE_PATH),
    "visible_rgb_color_count": unique_count,
    "palette_size": size_label,
    "dominant_color_families": [],  # à compléter à partir de l’image
    "palette_temperature": "unknown",
    "palette_structure": "unknown",
    "palette_contrast": "unknown",
    "review_status": "needs_visual_review",
}
display(example)